# 고정 이미지 encoder로 특징 벡터 추출

ResNet18 **512차원**, DINOv2-Small **384차원**, ConvNeXt-Tiny **768차원**을 모델별로 저장해.
분류·학습·파라미터 업데이트 없이 이미지를 벡터로 바꿔. sample_id로 팀원의 센서 특징과 연결하면 돼.
각 모델 셀을 하나씩 실행하거나, 설정에서 병렬 모드를 선택할 수 있어. 기본은 GPU 한 장에서 순차 처리야.

**서버의 개인 Python 커널을 선택해.** 실제 이미지·정답·벡터를 출력하거나 채팅에 올리지 않아.


## 1. 커널 확인

이 셀도 계속 대기하면 코드 문제가 아니라 커널 준비/연결부터 확인해야 해.


In [ ]:
print("커널 연결 정상", flush=True)

## 2. 경로와 실행 설정

기본은 Train·Validation 두 split이야. Test가 필요하면 SPLITS에 직접 추가해.
새 실행 폴더를 사용하므로 기존 결과는 보존해. 이 설정 셀을 다시 실행하면 새 폴더명이 정해져.
RUN_PARALLEL=False면 아래 모델별 셀을 순서대로 실행해.
RUN_PARALLEL=True면 모델별 셀은 건너뛰고 마지막 병렬 셀에서 최대 두 모델을 동시에 실행해.


In [ ]:
import os
import sys
import uuid
from datetime import datetime
from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / "image_features",
    Path("/home/ubuntu/seonghyeon/image_features"),
]
ROOT = next((p for p in candidates if (p / "encoders.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("image_features 폴더 전체를 열고 다시 실행해줘.")
sys.path.insert(0, str(ROOT))
DATA_DIR = Path(os.environ.get("DATA_DIR", "/home/ubuntu/data/battery_data_light"))
SPLITS = ("train", "validation")
RUN_PARALLEL = False
DEVICE = "cuda"
BATCH_SIZE = 32 if not RUN_PARALLEL else 16
WORKERS = 4 if not RUN_PARALLEL else 2
OUTPUT_ROOT = (
    ROOT
    / "outputs"
    / ("features_" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
)
print("결과 저장 위치:", OUTPUT_ROOT)

## 3. 패키지·GPU·입력 경로 확인

import는 첫 실행에서 시간이 걸릴 수 있어. 모델 다운로드나 이미지 읽기는 아직 하지 않아.
GPU 사용 여부는 실행 전에 팀원과 확인해. 개인 환경의 torch/CUDA를 임의로 교체하지 마.


In [ ]:
print("PyTorch와 추출 모듈을 불러오는 중...", flush=True)


In [ ]:
import torch
from extract_all import run_all
from extract_features import extract

if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA가 없어. 서버의 개인 Python 커널을 선택해줘.")
for split in SPLITS:
    if not (DATA_DIR / split / (split + ".csv")).is_file():
        raise RuntimeError("설정한 split의 CSV 경로를 확인해줘.")
print(
    {
        "python": sys.executable,
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "parallel": RUN_PARALLEL,
        "batch_size": BATCH_SIZE,
    }
)

## 4. 합성 이미지로 전처리 확인

RGB, 비율 유지 letterbox 224, ImageNet 정규화야. 실제 온도로 환산하지 않아.
기본 center crop 대신 이미지 전체를 보존하는 전처리라는 점을 실험 기록에 남겨.


In [ ]:
from encoders import preprocess
from PIL import Image

example = preprocess(Image.new("RGB", (80, 200), (180, 60, 20)))
assert example.shape == (3, 224, 224) and torch.isfinite(example).all()
print({"shape": tuple(example.shape), "dtype": str(example.dtype), "real_image_displayed": False})
del example

## 5. 모델별 추출 함수

각 split을 읽어 이미지당 하나의 벡터를 저장해. 아래 함수 자체는 아직 추출을 시작하지 않아.
첫 모델 실행에서는 공개 사전학습 가중치를 다운로드할 수 있어. 실제 이미지는 외부로 전송하지 않아.

같은 셀을 다시 실행해서 출력 폴더가 이미 있으면 split_retry_임의코드 폴더에 새로 추출해. 기존 완료·실패 결과는 보존하고 실제 저장 경로를 표시해.


In [ ]:
def extract_model_splits(model_name):
    import uuid

    reports = []
    for split in SPLITS:
        split_output = OUTPUT_ROOT / model_name / split
        if split_output.exists():
            split_output = split_output.with_name(split + "_retry_" + uuid.uuid4().hex[:8])
        print("결과 저장 위치:", split_output, flush=True)
        report = extract(
            model_name=model_name,
            csv_path=DATA_DIR / split / (split + ".csv"),
            image_root=DATA_DIR / split,
            output_dir=split_output,
            batch_size=BATCH_SIZE,
            workers=WORKERS,
            device=DEVICE,
        )
        reports.append({k: report[k] for k in ("model", "rows", "dimension", "seconds", "status")})
    return reports

## 6. ResNet18

fc를 제외한 512차원 특징이야. 학습하지 않고 eval/inference_mode로 추출해.


In [ ]:
if not RUN_PARALLEL:
    resnet18_reports = extract_model_splits("resnet18")
    print(resnet18_reports)
else:
    print("병렬 모드이므로 여기서는 실행하지 않아.")

## 7. DINOv2-Small

마지막 층 CLS token 384차원이야. transformers와 safetensors가 필요해. 학습하지 않고 eval/inference_mode로 추출해.


In [ ]:
if not RUN_PARALLEL:
    dinov2_reports = extract_model_splits("dinov2")
    print(dinov2_reports)
else:
    print("병렬 모드이므로 여기서는 실행하지 않아.")

## 8. ConvNeXt-Tiny

마지막 Linear만 제외하고 LayerNorm은 유지한 768차원 특징이야. 학습하지 않고 eval/inference_mode로 추출해.


In [ ]:
if not RUN_PARALLEL:
    convnext_tiny_reports = extract_model_splits("convnext_tiny")
    print(convnext_tiny_reports)
else:
    print("병렬 모드이므로 여기서는 실행하지 않아.")

## 9. 선택: 두 모델씩 병렬 실행

RUN_PARALLEL=True인 경우에만 실행돼. 같은 GPU를 공유하므로 메모리 부족이 날 수 있고,
순차보다 빨라진다는 보장은 없어. 병렬 실행에 문제가 있으면 README의 CLI 명령을 사용해.
실패한 모델을 자동 재실행하지 않고 이미 성공한 출력은 보존해.


In [ ]:
if RUN_PARALLEL:
    for split in SPLITS:
        parallel_reports = run_all(
            csv_path=DATA_DIR / split / (split + ".csv"),
            image_root=DATA_DIR / split,
            output_dir=OUTPUT_ROOT / ("parallel_" + split),
            batch_size=BATCH_SIZE,
            workers=WORKERS,
            device=DEVICE,
            parallel=2,
        )
        print(
            [
                {k: r[k] for k in ("model", "rows", "dimension", "seconds", "status")}
                for r in parallel_reports
            ]
        )
else:
    print("순차 모드: 위 모델별 셀의 결과를 사용해.")

## 10. 저장 결과 확인과 팀원 전달

metadata의 status=complete이고, features.npy의 행 수와 sample_ids.csv의 ID 수가 같아야 해.
팀원에게는 벡터·ID·metadata 세 파일을 같이 전달해. sample_id로 센서 데이터에 연결해.
원본 벡터나 ID 목록을 노트북 출력/채팅에 노출하지 않아.


In [ ]:
import json

import numpy as np

for metadata_path in sorted(OUTPUT_ROOT.rglob("metadata.json")):
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if metadata["status"] == "complete":
        vectors = np.load(metadata_path.parent / "features.npy", mmap_mode="r", allow_pickle=False)
        assert vectors.shape == (metadata["rows"], metadata["dimension"])
        print(
            {
                "model": metadata["model"],
                "shape": tuple(vectors.shape),
                "status": metadata["status"],
                "seconds": round(metadata["seconds"], 1),
            }
        )
    else:
        print({"model": metadata["model"], "status": metadata["status"]})